# POSG Validation — SQL + NoSQL (Phase 15A + 15B combined)

Combines `POSG_SQL-2.ipynb` and `POSG_NOSQL.ipynb` into one clean setup + test notebook.
Dead-end retries and two bugs from the originals are removed:
- NoSQL checkpoint symlinks were accidentally left pointing at the SQL checkpoints in one exploratory cell — fixed here.
- The per-database MongoDB loading cell hardcoded `"concert_singer"` regardless of the loop variable — replaced with `convert_all()` (loads all 166 databases correctly).

See `docs/phase15_posg_findings.md` for the full write-up of what these tests found.

**Status as of last run**: SQL dev-split (hard-only) EX 63.3% vs 60.0% greedy. NoSQL train-split EX 76.7% vs 73.3% greedy.
**Pending in this notebook**: SQL full-difficulty (non `--hard`) EX — the number comparable to the plan's >82% target.

## 1. Clone repo + install dependencies

In [ ]:
!git clone https://github.com/kethansplunk/Codegen.git
%cd Codegen
!pip install -q torch transformers peft sqlparse pyyaml FlagEmbedding chromadb openai python-dotenv pymongo

## 2. Mount Drive and load checkpoints (both tracks)

Loads SAR + Generator checkpoints for **both** SQL and NoSQL from Drive. If your Drive layout differs from `codegen/checkpoints/{sar,generator}_{sql,nosql}`, adjust `DRIVE` below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/codegen'
os.makedirs('models', exist_ok=True)

for name in ['sar_sql', 'generator_sql', 'sar_nosql', 'generator_nosql']:
    dst = f'models/{name}'
    if not os.path.exists(dst):
        os.symlink(f'{DRIVE}/checkpoints/{name}', dst)

!ls -la models/sar_sql models/generator_sql models/sar_nosql models/generator_nosql

In [ ]:
# Safety net: force sar.backend to memory regardless of whether the phase/15-posg
# PR has been merged into main yet. ChromaDB's PersistentClient can't open an index
# over a Google Drive FUSE mount, so this avoids that failure mode entirely.
text = open('configs/config.yaml').read()
text = text.replace('backend: chroma', 'backend: memory')
open('configs/config.yaml', 'w').write(text)
!grep -A1 "^sar:" configs/config.yaml | head -3

## 3. Spider SQLite databases

Needed for SQL EX scoring, and as the source data for the MongoDB conversion below. Uploaded once as a zip to Drive (see `docs/phase15_posg_findings.md` for how it was built, from a sibling local project).

In [ ]:
!cp /content/drive/MyDrive/codegen/checkpoints/spider_database.zip /content/Codegen/
!unzip -q /content/Codegen/spider_database.zip -d /content/Codegen/Data/Spider/
!ls /content/Codegen/Data/Spider/database | wc -l

## 4. MongoDB setup (needed for NoSQL EX)

**Important**: `Data/mongodb/*.json` schema-cache files are git-tracked from an earlier run on a different machine. `convert_all()` treats their existence as "already converted" and will silently skip real data insertion into this fresh session's empty `mongod` if we don't clear them first — this bit us once already (see the findings doc). The `shutil.rmtree` below is required, not optional.

In [ ]:
# Install and start mongod
!apt-get install -y mongodb >/dev/null 2>&1 || (curl -fsSL https://pgp.mongodb.com/server-7.0.asc | sudo gpg -o /usr/share/keyrings/mongodb-server-7.0.gpg --dearmor && echo "deb [signed-by=/usr/share/keyrings/mongodb-server-7.0.gpg arch=amd64] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 multiverse" | sudo tee /etc/apt/sources.list.d/mongodb-org-7.0.list && apt-get update -qq && apt-get install -y mongodb-org)
!mkdir -p /data/db
import subprocess, time
subprocess.Popen(["mongod", "--dbpath", "/data/db", "--bind_ip", "127.0.0.1"])
time.sleep(5)
!mongosh --eval "db.version()" 

In [ ]:
import shutil
shutil.rmtree('Data/mongodb', ignore_errors=True)   # force a real reconversion, see note above

from src.mongodb_converter import convert_all
convert_all(
    db_root="Data/Spider/database",
    fk_graph_dir="Data/fk_graphs",
    schema_cache_dir="Data/mongodb",
)

In [ ]:
# Verify real data landed (not just schema cache) before trusting any EX result
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017")
print(len(client.list_database_names()), client.list_database_names()[:10])
print("formula_1.drivers count:", client['formula_1']['drivers'].count_documents({}))   # must be > 0

## 5. SQL track — full dev-set smoke test (all difficulty)

`sql_dev_eval_full.json` (1034 Spider dev questions, real SchemaLinker `key_fields`) was rebuilt locally on Mac after the Colab runtime that originally built it disconnected. Upload it to Drive from your Mac first, then pull it in here. This run (no `--hard` filter) is the one comparable to the plan's >82% EX target — we only have the hard-only number (63.3%) so far.

In [ ]:
!cp /content/drive/MyDrive/codegen/sql_dev_eval_full.json Data/cot_data/sql_dev_eval_full.json
!python -m scripts.run_posg_sql --smoke_test --n 30 --data Data/cot_data/sql_dev_eval_full.json

## 6. NoSQL track — train-split smoke test (all difficulty)

Already validated: EX 76.7% (POSG) vs 73.3% (greedy) on this exact command. Re-run here to confirm reproducibility in a fresh session, or change `--n`/`--seed` for a different sample.

In [ ]:
!python -m scripts.run_posg_nosql --smoke_test --n 30

## 7. Optional — DeepSeek API key

Only needed if you want to rebuild a dev-eval-set file (`scripts/build_dev_eval_set.py`) or run the `--question` single-question mode — the smoke tests above reuse pre-computed `key_fields` and never call DeepSeek. **Never commit this cell with a real key filled in.**

In [ ]:
with open('.env', 'w') as f:
    f.write('DEEPSEEK_API_KEY=your_actual_key_here\n')